In [3]:
# Configuration AWS
BEDROCK_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
AWS_REGION = "us-east-1"

# OpenSearch Configuration
OPENSEARCH_URL = "https://pyx7grd5myluqvsirhji.ap-south-1.aoss.amazonaws.com"
OPENSEARCH_HOST ="pyx7grd5myluqvsirhji.ap-south-1.aoss.amazonaws.com"
INDEX_NAME = "rag-bedrock-index"

In [8]:
# Imports
import os
import boto3
from opensearchpy import RequestsHttpConnection, AWSV4SignerAuth

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings
from langchain_aws import ChatBedrock

In [9]:
# OpenSearch Auth
credentials = boto3.Session().get_credentials()

awsauth = AWSV4SignerAuth(
    credentials,
    AWS_REGION,
    "es"
)

In [5]:
# Initialize Embeddings
embedding = BedrockEmbeddings(
    model_id=BEDROCK_EMBEDDING_MODEL_ID,
    region_name=AWS_REGION
)

In [6]:
# Load faiss store
faiss_store = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

In [7]:
docs = list(faiss_store.docstore._dict.values())

print("Documents extracted:", len(docs))

Documents extracted: 1808


In [13]:
vector_store = OpenSearchVectorSearch(
    opensearch_url=f"https://{OPENSEARCH_HOST}",
    index_name=INDEX_NAME,
    embedding_function=embedding,
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    bulk_size=2000
)

In [20]:
vector_store.add_documents(docs)

print("Migration complete")

AuthorizationException: AuthorizationException(403, 'Forbidden')